In [3]:
import pandas as pd
import numpy as np
import optuna
from sklearn.model_selection import LeaveOneOut, StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler
from sklearn.feature_selection import SelectKBest, f_classif, SelectFromModel
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
import xgboost as xgb
import warnings
from IPython.display import clear_output

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING) # Silencia os logs gigantes do Optuna

print("="*80)
print("MOTOR DE EXPERIMENTAÇÃO V1: OTIMIZAÇÃO BAYESIANA E DETECÇÃO DE P-HACKING")
print("="*80)

CAMINHO_MATRIZ = '/workspaces/EyeTracking/data/processed/matriz_features_ml.csv'
df = pd.read_csv(CAMINHO_MATRIZ)

y = (df['Grupo'] == 'TEA').astype(int)
X_total = df.drop(columns=['Paciente', 'Grupo'])

# ---------------------------------------------------------
# CONFIGURAÇÃO DO SUBCONJUNTO (Alvo atual: Raiva Humano + Todas as Features)
# ---------------------------------------------------------
colunas_alvo = [col for col in X_total.columns if 'Humano' in col and 'Raiva' in col]
vars_globais = [v for v in ['Velocidade_Sacadica_Media', 'Num_Total_Fixacoes', 'Duracao_Media_Fixacao_ms', 
                            'Area_Dispersao_ConvexHull', 'Entropia_Shannon_Olhar', 'Num_Transicoes_Olho_Boca',
                            'Pupila_Reatividade_Global', 'Fuga_Visual_Global_%', 'EMI_Global', 'TTFF_Medio_Olhos_ms'] if v in X_total.columns]
X = X_total[list(set(colunas_alvo + vars_globais))]

# ---------------------------------------------------------
# FUNÇÃO DA MÉTRICA PERSONALIZADA (Equação Clínica)
# ---------------------------------------------------------
def calcular_score_clinico(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    cm = confusion_matrix(y_true, y_pred)
    
    if cm.shape == (2, 2):
        vp, fn, vn, fp = cm[1][1], cm[1][0], cm[0][0], cm[0][1]
        sensibilidade = vp / (vp + fn) if (vp + fn) > 0 else 0
        especificidade = vn / (vn + fp) if (vn + fp) > 0 else 0
    else:
        sensibilidade, especificidade = 0, 0
        
    score = (0.45 * sensibilidade) + (0.30 * acc) + (0.15 * f1) + (0.10 * especificidade)
    return score, acc, sensibilidade, especificidade, f1

# ---------------------------------------------------------
# O CÉREBRO OPTUNA (Busca Inteligente do Melhor Pipeline)
# ---------------------------------------------------------
def objective(trial):
    # 1. Espaço dos Scalers
    scaler_name = trial.suggest_categorical("scaler", ["Standard", "Robust", "MinMax"])
    if scaler_name == "Standard": scaler = StandardScaler()
    elif scaler_name == "Robust": scaler = RobustScaler()
    else: scaler = MinMaxScaler()
        
    # 2. Espaço dos Seletores
    k_features = trial.suggest_int("k_features", 3, 15)
    selector_name = trial.suggest_categorical("selector", ["KBest", "ExtraTrees"])
    
    if selector_name == "KBest":
        selector = SelectKBest(f_classif, k=k_features)
    else:
        et = ExtraTreesClassifier(n_estimators=50, random_state=42)
        selector = SelectFromModel(et, max_features=k_features)

    # 3. Espaço dos Classificadores e Hiperparâmetros
    model_name = trial.suggest_categorical("model", ["LogReg", "SVM_Linear", "SVM_RBF", "RandomForest", "XGBoost"])
    
    if model_name == "LogReg":
        C_lr = trial.suggest_float("C_lr", 0.01, 10.0, log=True)
        modelo = LogisticRegression(C=C_lr, class_weight='balanced', random_state=42, max_iter=500)
    elif model_name == "SVM_Linear":
        C_svml = trial.suggest_float("C_svml", 0.01, 10.0, log=True)
        modelo = SVC(kernel='linear', C=C_svml, class_weight='balanced', random_state=42)
    elif model_name == "SVM_RBF":
        C_svmr = trial.suggest_float("C_svmr", 0.1, 20.0, log=True)
        modelo = SVC(kernel='rbf', C=C_svmr, class_weight='balanced', random_state=42)
    elif model_name == "RandomForest":
        depth_rf = trial.suggest_int("depth_rf", 2, 5)
        modelo = RandomForestClassifier(max_depth=depth_rf, class_weight='balanced', random_state=42)
    else:
        depth_xgb = trial.suggest_int("depth_xgb", 1, 3)
        lr_xgb = trial.suggest_float("lr_xgb", 0.01, 0.2, log=True)
        modelo = xgb.XGBClassifier(max_depth=depth_xgb, learning_rate=lr_xgb, scale_pos_weight=(sum(y==0)/sum(y==1)), eval_metric='logloss', random_state=42)

    pipeline = Pipeline([('scaler', scaler), ('selector', selector), ('classifier', modelo)])
    
    # Avaliação rápida para o Optuna guiar a busca (3-Fold Estratificado)
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    try:
        y_pred_cv = cross_val_predict(pipeline, X, y, cv=cv)
        score, _, _, _, _ = calcular_score_clinico(y, y_pred_cv)
        return score
    except:
        return 0.0

print("Fase 1: Otimização Bayesiana com Optuna (Procurando a melhor arquitetura)...")
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30) # 30 tentativas inteligentes é suficiente para esse espaço

best_params = study.best_params
print(f"\nMelhor Pipeline Encontrado: {best_params['model']} com {best_params['scaler']} e {best_params['selector']} (K={best_params['k_features']})")

# ---------------------------------------------------------
# RECONSTRUÇÃO DO MELHOR PIPELINE PARA O TESTE FINAL
# ---------------------------------------------------------
if best_params['scaler'] == "Standard": best_scaler = StandardScaler()
elif best_params['scaler'] == "Robust": best_scaler = RobustScaler()
else: best_scaler = MinMaxScaler()

if best_params['selector'] == "KBest": best_selector = SelectKBest(f_classif, k=best_params['k_features'])
else: best_selector = SelectFromModel(ExtraTreesClassifier(n_estimators=50, random_state=42), max_features=best_params['k_features'])

if best_params['model'] == "LogReg": best_model = LogisticRegression(C=best_params['C_lr'], class_weight='balanced', max_iter=500)
elif best_params['model'] == "SVM_Linear": best_model = SVC(kernel='linear', C=best_params['C_svml'], class_weight='balanced')
elif best_params['model'] == "SVM_RBF": best_model = SVC(kernel='rbf', C=best_params['C_svmr'], class_weight='balanced')
elif best_params['model'] == "RandomForest": best_model = RandomForestClassifier(max_depth=best_params['depth_rf'], class_weight='balanced')
else: best_model = xgb.XGBClassifier(max_depth=best_params['depth_xgb'], learning_rate=best_params['lr_xgb'], scale_pos_weight=(sum(y==0)/sum(y==1)), eval_metric='logloss')

def rodar_loocv(semente_atual):
    if hasattr(best_model, 'random_state'): best_model.set_params(random_state=semente_atual)
    if best_params['selector'] == "ExtraTrees": best_selector.estimator.set_params(random_state=semente_atual)
        
    pipe_final = Pipeline([('scaler', best_scaler), ('selector', best_selector), ('classifier', best_model)])
    previsoes = []
    
    for train_idx, test_idx in LeaveOneOut().split(X):
        pipe_final.fit(X.iloc[train_idx], y.iloc[train_idx])
        previsoes.append(pipe_final.predict(X.iloc[test_idx])[0])
        
    return calcular_score_clinico(y, previsoes)

# ---------------------------------------------------------
# O DUELO: VERDADE CIENTÍFICA vs P-HACKING
# ---------------------------------------------------------
print("\nFase 2: Executando a Prova Cega (LOOCV)...")

# A. A Verdade Científica (Semente Engessada)
_, acc_42, sens_42, esp_42, _ = rodar_loocv(42)

# B. A Busca pelo Milagre (Seed Hunting 1-100)
# Apenas rodamos o seed hacking se o modelo for instável (Árvores) ou tiver seletor baseado em árvores
modelos_instaveis = ["RandomForest", "XGBoost"]
precisa_seed_hunting = best_params['model'] in modelos_instaveis or best_params['selector'] == "ExtraTrees"

acc_hacked, sens_hacked, esp_hacked, melhor_semente = acc_42, sens_42, esp_42, 42

if precisa_seed_hunting:
    print("Fase 3: Otimizando ao extremo (Caça à Semente para P-Hacking)...")
    melhor_score_hack = 0
    for seed in range(1, 101):
        score, acc_s, sens_s, esp_s, _ = rodar_loocv(seed)
        if score > melhor_score_hack:
            melhor_score_hack, acc_hacked, sens_hacked, esp_hacked, melhor_semente = score, acc_s, sens_s, esp_s, seed

clear_output(wait=True)
print("="*90)
print(f"{'RELATÓRIO DE DEFESA: O IMPACTO DA ESTABILIDADE METODOLÓGICA':^90}")
print("="*90)
print(f"Arquitetura Vencedora (Optuna): {best_params['model']} | Scaler: {best_params['scaler']} | Seletor: {best_params['selector']} (K={best_params['k_features']})")
print("-" * 90)
print(f"RESULTADO REPRODUTÍVEL (Semente 42 - A Verdade Científica):")
print(f"   Acurácia: {acc_42*100:.1f}% | Sensibilidade (TEA): {sens_42*100:.1f}% | Especificidade (Ctrl): {esp_42*100:.1f}%")
print("-" * 90)

if precisa_seed_hunting:
    print(f"RESULTADO P-HACKING (Melhor Semente: {melhor_semente} - A Ilusão Estatística):")
    print(f"   Acurácia: {acc_hacked*100:.1f}% | Sensibilidade (TEA): {sens_hacked*100:.1f}% | Especificidade (Ctrl): {esp_hacked*100:.1f}%")
else:
    print("O modelo vencedor é determinístico (SVM/LogReg). Não há margem para P-Hacking de semente.")
print("="*90)

               RELATÓRIO DE DEFESA: O IMPACTO DA ESTABILIDADE METODOLÓGICA                
Arquitetura Vencedora (Optuna): LogReg | Scaler: Robust | Seletor: KBest (K=12)
------------------------------------------------------------------------------------------
RESULTADO REPRODUTÍVEL (Semente 42 - A Verdade Científica):
   Acurácia: 65.8% | Sensibilidade (TEA): 58.8% | Especificidade (Ctrl): 71.4%
------------------------------------------------------------------------------------------
O modelo vencedor é determinístico (SVM/LogReg). Não há margem para P-Hacking de semente.


In [1]:
import os
import glob
import pandas as pd
import numpy as np
import optuna
from sklearn.model_selection import LeaveOneOut, StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler
from sklearn.feature_selection import SelectKBest, f_classif, SelectFromModel
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
import xgboost as xgb
import warnings
from IPython.display import clear_output

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

print("="*80)
print("A FUSÃO FINAL: MATRIZ DE MARKOV + VARIÁVEIS GLOBAIS NO OPTUNA")
print("="*80)

# ---------------------------------------------------------
# 1. GERAÇÃO RÁPIDA DA MATRIZ DE MARKOV
# ---------------------------------------------------------
PASTA_CSV = '/workspaces/EyeTracking/data/csv/'
arquivos = glob.glob(os.path.join(PASTA_CSV, '*.csv'))
lista_markov = []
estados_possiveis = ['Olhos', 'Nariz', 'Boca', 'Rosto_Fundo', 'Fora', 'Desconhecido']

print("Lendo CSVs para extrair a Geometria de Scanpath (Raiva)...")
for caminho in arquivos:
    nome_puro = os.path.basename(caminho).replace('.csv', '')
    try:
        df = pd.read_csv(caminho)
        df_alvo = df[(df['Fase_Estimulo'] == 'Exposicao_Face') & 
                     (df['Tipo_Estimulo'] == 'Humano') & 
                     (df['Emocao'] == 'Raiva')].copy()
        
        if len(df_alvo) == 0: continue
            
        aois = ['AOI_Ambos_Olhos', 'AOI_Nariz', 'AOI_Boca', 'AOI_Rosto', 'AOI_Fora']
        for aoi in aois: df_alvo[aoi] = df_alvo[aoi].fillna(False).astype(bool)
            
        condicoes = [
            df_alvo['AOI_Ambos_Olhos'] == True, df_alvo['AOI_Boca'] == True,
            df_alvo['AOI_Nariz'] == True, df_alvo['AOI_Rosto'] == True, df_alvo['AOI_Fora'] == True
        ]
        df_alvo['Estado_Atual'] = np.select(condicoes, ['Olhos', 'Boca', 'Nariz', 'Rosto_Fundo', 'Fora'], default='Desconhecido')
        
        mudancas = df_alvo[df_alvo['Estado_Atual'] != df_alvo['Estado_Atual'].shift(1)]
        sequencia = mudancas['Estado_Atual'].tolist()
        
        paciente_features = {'Paciente': nome_puro}
        for origem in estados_possiveis:
            for destino in estados_possiveis: paciente_features[f"Trans_{origem}_para_{destino}"] = 0.0
                
        transicoes = pd.DataFrame({'Origem': sequencia[:-1], 'Destino': sequencia[1:]})
        if len(transicoes) > 0:
            matriz_prob = pd.crosstab(transicoes['Origem'], transicoes['Destino'], normalize='index')
            for origem in matriz_prob.index:
                for destino in matriz_prob.columns:
                    paciente_features[f"Trans_{origem}_para_{destino}"] = matriz_prob.loc[origem, destino] * 100
                    
        lista_markov.append(paciente_features)
    except: pass

df_markov = pd.DataFrame(lista_markov)

# ---------------------------------------------------------
# 2. FUSÃO COM A MATRIZ GLOBAL
# ---------------------------------------------------------
CAMINHO_MATRIZ = '/workspaces/EyeTracking/data/processed/matriz_features_ml.csv'
df_features = pd.read_csv(CAMINHO_MATRIZ)
df_fusion = pd.merge(df_features, df_markov, on='Paciente', how='inner')

y = (df_fusion['Grupo'] == 'TEA').astype(int)

# Selecionamos: Variáveis Globais de Pupila/Exploração + As rotas de Markov
vars_globais = [v for v in ['Velocidade_Sacadica_Media', 'Num_Total_Fixacoes', 'Duracao_Media_Fixacao_ms', 
                            'Area_Dispersao_ConvexHull', 'Entropia_Shannon_Olhar', 'Num_Transicoes_Olho_Boca',
                            'Pupila_Reatividade_Global', 'Fuga_Visual_Global_%', 'EMI_Global', 'TTFF_Medio_Olhos_ms'] if v in df_fusion.columns]
rotas_markov = [col for col in df_fusion.columns if col.startswith('Trans_')]

colunas_alvo = list(set(vars_globais + rotas_markov))
X = df_fusion[colunas_alvo].drop(columns=['Trans_Desconhecido_para_Desconhecido'], errors='ignore')
X = X.loc[:, (X != 0).any(axis=0)] # Remove colunas sempre nulas

print(f"Fusão completa. Iniciando Optuna com {X.shape[1]} features de alto nível...")

# ---------------------------------------------------------
# 3. MOTOR OPTUNA (Cérebro Clínico)
# ---------------------------------------------------------
def calcular_score_clinico(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    cm = confusion_matrix(y_true, y_pred)
    if cm.shape == (2, 2):
        vp, fn, vn, fp = cm[1][1], cm[1][0], cm[0][0], cm[0][1]
        sens = vp / (vp + fn) if (vp + fn) > 0 else 0
        esp = vn / (vn + fp) if (vn + fp) > 0 else 0
    else: sens, esp = 0, 0
    return (0.45 * sens) + (0.30 * acc) + (0.15 * f1) + (0.10 * esp), acc, sens, esp, f1

def objective(trial):
    scaler_name = trial.suggest_categorical("scaler", ["Standard", "Robust", "MinMax"])
    scaler = StandardScaler() if scaler_name == "Standard" else RobustScaler() if scaler_name == "Robust" else MinMaxScaler()
        
    k_features = trial.suggest_int("k_features", 5, 20)
    selector_name = trial.suggest_categorical("selector", ["KBest", "ExtraTrees"])
    if selector_name == "KBest": selector = SelectKBest(f_classif, k=k_features)
    else: selector = SelectFromModel(ExtraTreesClassifier(n_estimators=50, random_state=42), max_features=k_features)

    model_name = trial.suggest_categorical("model", ["LogReg", "SVM_Linear", "SVM_RBF", "RandomForest", "XGBoost"])
    if model_name == "LogReg":
        modelo = LogisticRegression(C=trial.suggest_float("C_lr", 0.01, 10.0, log=True), class_weight='balanced', random_state=42, max_iter=500)
    elif model_name == "SVM_Linear":
        modelo = SVC(kernel='linear', C=trial.suggest_float("C_svml", 0.01, 10.0, log=True), class_weight='balanced', random_state=42)
    elif model_name == "SVM_RBF":
        modelo = SVC(kernel='rbf', C=trial.suggest_float("C_svmr", 0.1, 20.0, log=True), class_weight='balanced', random_state=42)
    elif model_name == "RandomForest":
        modelo = RandomForestClassifier(max_depth=trial.suggest_int("depth_rf", 2, 5), class_weight='balanced', random_state=42)
    else:
        modelo = xgb.XGBClassifier(max_depth=trial.suggest_int("depth_xgb", 1, 3), learning_rate=trial.suggest_float("lr_xgb", 0.01, 0.2, log=True), scale_pos_weight=(sum(y==0)/sum(y==1)), eval_metric='logloss', random_state=42)

    pipeline = Pipeline([('scaler', scaler), ('selector', selector), ('classifier', modelo)])
    try:
        y_pred_cv = cross_val_predict(pipeline, X, y, cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42))
        return calcular_score_clinico(y, y_pred_cv)[0]
    except: return 0.0

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=40)
best = study.best_params

# ---------------------------------------------------------
# 4. AVALIAÇÃO FINAL BLINDADA (LOOCV)
# ---------------------------------------------------------
scaler = StandardScaler() if best['scaler'] == "Standard" else RobustScaler() if best['scaler'] == "Robust" else MinMaxScaler()
selector = SelectKBest(f_classif, k=best['k_features']) if best['selector'] == "KBest" else SelectFromModel(ExtraTreesClassifier(n_estimators=50, random_state=42), max_features=best['k_features'])

if best['model'] == "LogReg": best_model = LogisticRegression(C=best['C_lr'], class_weight='balanced', max_iter=500)
elif best['model'] == "SVM_Linear": best_model = SVC(kernel='linear', C=best['C_svml'], class_weight='balanced')
elif best['model'] == "SVM_RBF": best_model = SVC(kernel='rbf', C=best['C_svmr'], class_weight='balanced')
elif best['model'] == "RandomForest": best_model = RandomForestClassifier(max_depth=best['depth_rf'], class_weight='balanced')
else: best_model = xgb.XGBClassifier(max_depth=best['depth_xgb'], learning_rate=best['lr_xgb'], scale_pos_weight=(sum(y==0)/sum(y==1)), eval_metric='logloss')

pipe_final = Pipeline([('scaler', scaler), ('selector', selector), ('classifier', best_model)])

previsoes = []
for train_idx, test_idx in LeaveOneOut().split(X):
    pipe_final.fit(X.iloc[train_idx], y.iloc[train_idx])
    previsoes.append(pipe_final.predict(X.iloc[test_idx])[0])
    
_, acc_42, sens_42, esp_42, _ = calcular_score_clinico(y, previsoes)

clear_output(wait=True)
print("="*90)
print(f"{'VEREDITO FINAL DA DISSERTAÇÃO: MATRIZ MARKOV + VARIÁVEIS GLOBAIS':^90}")
print("="*90)
print(f"Arquitetura Ótima: {best['model']} | Scaler: {best['scaler']} | Seletor: {best['selector']} (K={best['k_features']})")
print("-" * 90)
print(f"RESULTADO REPRODUTÍVEL (A Verdade Científica Absoluta):")
print(f"   Acurácia: {acc_42*100:.1f}% | Sensibilidade (TEA): {sens_42*100:.1f}% | Especificidade (Ctrl): {esp_42*100:.1f}%")
print("="*90)

pipe_final.fit(X, y)
if best['selector'] == "KBest": features_ativas = X.columns[pipe_final.named_steps['selector'].get_support()].tolist()
else: features_ativas = X.columns[pipe_final.named_steps['selector'].get_support()].tolist()

print("\n🔍 OS BIOMARCADORES HÍBRIDOS (A Assinatura Clínica Definitiva):")
for i, f in enumerate(features_ativas, 1): print(f"   {i}. {f.replace('Trans_', '').replace('_para_', ' ➔ ')}")

             VEREDITO FINAL DA DISSERTAÇÃO: MATRIZ MARKOV + VARIÁVEIS GLOBAIS             
Arquitetura Ótima: LogReg | Scaler: Standard | Seletor: ExtraTrees (K=15)
------------------------------------------------------------------------------------------
RESULTADO REPRODUTÍVEL (A Verdade Científica Absoluta):
   Acurácia: 57.9% | Sensibilidade (TEA): 52.9% | Especificidade (Ctrl): 61.9%

🔍 OS BIOMARCADORES HÍBRIDOS (A Assinatura Clínica Definitiva):
   1. Num_Transicoes_Olho_Boca
   2. Fora ➔ Olhos
   3. Area_Dispersao_ConvexHull
   4. Olhos ➔ Nariz
   5. Olhos ➔ Rosto_Fundo
   6. Pupila_Reatividade_Global
   7. Nariz ➔ Rosto_Fundo
   8. Rosto_Fundo ➔ Olhos
   9. Velocidade_Sacadica_Media
   10. Fora ➔ Rosto_Fundo
   11. Rosto_Fundo ➔ Fora
